## 01- Transformer Classifier Baseline with Extra Pitt Training data
In this notebook, we will do a baseline using a mix of classicial and neural method (Transformer-based sentence embeddings and neural classifier)

In [1]:
!pip install sentence-transformers

In [2]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
from sklearn.metrics import precision_recall_fscore_support, f1_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import cross_val_score


In [3]:
# train_fname = "../data/splits/train.csv"
# dev_fname = "../data/splits/dev.csv"

# new - load 50/50 data splits
train_fname = "../data/splits/train_50pc.csv"
dev_fname = "../data/splits/dev_50pc.csv"

train_df = pd.read_csv(train_fname)
dev_df = pd.read_csv(dev_fname)

print("Train len =", len(train_df))
print("Dev df =", len(dev_df))

Train len = 83
Dev df = 83


In [4]:
train_df.head()

,text,target
0,I think you see going on in that picture. We'l...,AD
1,I'm gonna give you a picture that has a lot of...,AD
2,Ah the sinks running over waters come on all f...,CN
3,"Well, my boys trying to get some cookies and t...",AD
4,I'd like you to do is just look at the picture...,CN


In [5]:
dev_df.head()

,text,target
0,"Everything you see going on in this picture, g...",CN
1,"Now you just look at the picture. Okay, this i...",CN
2,Tell me everything that you see going on in th...,AD
3,Girls and boys getting her one that stores sto...,AD
4,"All right, we have a few more tasks to go thro...",CN


In [6]:
# load extra data
extra_train_df = pd.read_csv("../data/splits/train_pitt.csv")
extra_train_df.head()

,text,target
0,little boy getting in the cookie jar .\nand th...,CN
1,well ‡ the sink's overflowing .\nand &-uh the ...,CN
2,"this picture ?\nwell, there's a kid on a stool...",AD
3,everything I see in action or everything I +.....,CN
4,there's a little boy or> there's a little gir...,AD


### Pre-processing
We will lowercase the text for better efficiency

In [7]:
train_df["text_lower"] = train_df["text"].apply(lambda text: text.lower().strip())
dev_df["text_lower"] = dev_df["text"].apply(lambda text: text.lower().strip())

In [8]:
extra_train_df["text_lower"] = extra_train_df["text"].apply(lambda text: text.lower().strip())

In [9]:
# stack up pre-train and train df

train_df = pd.concat([train_df, extra_train_df], ignore_index=True, axis=0)

# # test with Pitt train set only
# train_df = extra_train_df

print(len(train_df))
train_df.head()

455


,text,target,text_lower
0,I think you see going on in that picture. We'l...,AD,i think you see going on in that picture. we'l...
1,I'm gonna give you a picture that has a lot of...,AD,i'm gonna give you a picture that has a lot of...
2,Ah the sinks running over waters come on all f...,CN,ah the sinks running over waters come on all f...
3,"Well, my boys trying to get some cookies and t...",AD,"well, my boys trying to get some cookies and t..."
4,I'd like you to do is just look at the picture...,CN,i'd like you to do is just look at the picture...


### Apply Sentence Transformer

We can use Sentence Transformer to convert text to a fixed-length embedding vector

In [10]:
# vectorizer = SentenceTransformer("sentence-t5-large") # works best
vectorizer = SentenceTransformer("gtr-t5-large") # a general purpose model

In [11]:
corpus = train_df["text_lower"]
print("Creating text embeddings...")
X_train = vectorizer.encode(corpus)
print("Completed")

Creating text embeddings...
Completed


In [12]:
print(X_train.shape)

(455, 768)


### Train Neural Classifier

We can now train a neural logistic regression classifier using processed X_train

In [13]:
# # # X_train = vectorizer.fit_transform(stacked_train_df["text_lower"])
# y_train = train_df["target"]

# print("Training logistic classifier...")
# clf = LogisticRegression(random_state=42, max_iter=200).fit(X_train, y_train)
# # clf = LogisticRegression(random_state=42, 
# #                          max_iter=200,
# #                         ).fit(X_train, y_train)
# print("Completed")

In [14]:
y_train = train_df["target"]

print("Training neural classifier...")
# hidden_size = 500
hidden_size = 100

clf = MLPClassifier(solver="lbfgs",
                    alpha=1e-5,
                    # hidden_layer_sizes=(hidden_size, hidden_size, hidden_size),
                    hidden_layer_sizes=(hidden_size,),
                    # activation="relu",
                    # shuffle=True,
                    # learning_rate="constant",
                    learning_rate_init=0.0001,
                    max_iter=100, 
                    random_state=42
                   )
clf.fit(X_train, y_train)
print("Completed")

Training neural classifier...
Completed


In [15]:
# getting score on train set
train_score = clf.score(X_train, y_train)
print(f"Train accuracy = {train_score*100:.2f} %")

Train accuracy = 100.00 %


In [16]:
# get precision / recall / f1
y_pred = clf.predict(X_train)
print(classification_report(y_train, y_pred))

              precision    recall  f1-score   support

          AD       1.00      1.00      1.00       245
          CN       1.00      1.00      1.00       210

    accuracy                           1.00       455
   macro avg       1.00      1.00      1.00       455
weighted avg       1.00      1.00      1.00       455



### Get accuracy using cross validation

In [17]:
print("Performing cross-validation...")
scores = cross_val_score(clf, X_train, y_train, cv=10)
print("Completed")
print(scores)

Performing cross-validation...
Completed
[0.76086957 0.7826087  0.82608696 0.73913043 0.7173913  0.75555556
 0.86666667 0.8        0.8        0.77777778]


In [18]:
print("Average =", np.average(scores))

Average = 0.782608695652174


### Now we can evaluate on dev set

In [19]:
X_dev = vectorizer.encode(dev_df["text_lower"])
y_dev = dev_df["target"]
print("X_dev shape =", X_dev.shape)
print("y_dev len =", len(y_dev))

X_dev shape = (83, 768)
y_dev len = 83


In [20]:
dev_score = clf.score(X_dev, y_dev)
print(f"Dev accuracy = {dev_score*100:.2f} %")

Dev accuracy = 79.52 %


In [21]:
y_pred = clf.predict(X_dev)
print(classification_report(y_dev, y_pred))

              precision    recall  f1-score   support

          AD       0.80      0.85      0.82        47
          CN       0.79      0.72      0.75        36

    accuracy                           0.80        83
   macro avg       0.79      0.79      0.79        83
weighted avg       0.79      0.80      0.79        83



We can see, with a few changes:
- using extra train set from Pitt dataset
- change embedding model from "sentence-t5-large" to "gtr-t5-large"
- reduce precision of neural classifier (max_inter = 100, hidden_size = 100)

We can have:
- cross-validation average accuracy of 78%, consistent with dev set score 80%
- recall on dev set to 85%, which is not as high as classicial model with extra data (90%-95%), but seems to be promising.